# KneeXAI — GPU research runner for Kaggle

This notebook runs the repository's **existing** `scripts/run_research.py` on Kaggle GPU. It does not reimplement the metrics, so the resulting CSV follows the same validated contract used by Research Mode.

## Before running

1. In Kaggle Notebook settings, turn **Accelerator → GPU** on.
2. Attach a Kaggle Dataset containing this project source (without `.venv`, `.git`, and model weights is fine).
3. Attach a second private Kaggle Dataset containing the model files. For the selected model it must contain both the `.pth` checkpoint and matching `.onnx` file listed in the configuration cell.
4. For a thesis run, replace the demo manifest with a labelled JSON manifest for real images. Every `true_kl` and `split` must already be known; this notebook never guesses labels from folder names.
5. Enable Internet only if Kaggle needs to install `quantus` and ONNX packages.

The default values below use the six-image repository demo only as a technical check — not as scientific evidence.

In [ ]:
# Kaggle already provides CUDA-enabled PyTorch when GPU is enabled.
# Do NOT install torch from requirements.txt here: that file is intentionally CPU-oriented.
import sys
!{sys.executable} -m pip install -q quantus==0.6.0 onnx==1.17.0 onnxruntime==1.20.1 opencv-python-headless==4.10.0.84

import torch
assert torch.cuda.is_available(), (
    'CUDA is unavailable. In Kaggle open Notebook options and select Accelerator → GPU, ' 
    'then restart the session and run this cell again.'
)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# CHANGE THESE THREE PATHS to the Kaggle Dataset slugs that you attached.
PROJECT_SOURCE = Path('/kaggle/input/kneexai-project')
MODELS_SOURCE = Path('/kaggle/input/kneexai-models')

# Select exactly one final ordinal-BCE model: 'resnet50' or 'densenet121'.
MODEL_NAME = 'resnet50'

# The included 150-image labelled test subset has 30 images per KL grade and supports
# the 10-per-grade sensitivity profile. Replace this only with another labelled manifest.
# Relative image paths are resolved from the copied project root.
MANIFEST_SOURCE = PROJECT_SOURCE / 'inputs' / 'test' / 'test_manifest.json'

# Choose one profile:
# - 'full': all five metrics, for the final selected configuration;
# - 'faithfulness_sensitivity': three separate CSVs at 224, 1024 and 2048 spatial RGB groups;
# - 'max_sensitivity_sensitivity': raw and independently max-normalised attribution maps.
EXPERIMENT_PROFILE = 'faithfulness_sensitivity'

# First run: False. To resume, keep this prefix and every setting below unchanged.
RESUME = False
RUN_ID_PREFIX = f'{MODEL_NAME}-xai-sensitivity-20260924'  # Change only before a new experiment.

# Use 10 labelled images per KL grade for a balanced 50-image experiment.
# Set SAMPLES_PER_KL = None only when you deliberately want the whole manifest.
LIMIT = None                 # e.g. 1 only for a technical smoke test
SAMPLES_PER_KL = 10
FAITHFULNESS_RUNS = 30 if EXPERIMENT_PROFILE != 'full' else 50
MAX_SENSITIVITY_SAMPLES = 8
FINAL_FAITHFULNESS_SUBSET_SIZE = 224  # Set after reviewing the sensitivity results.
SEED = 42
METHODS = ['Grad-CAM', 'Grad-CAM++', 'Eigen-CAM', 'SmoothGrad', 'Integrated Gradients']

# Attach these exact final checkpoint/ONNX pairs in the Kaggle models dataset.
# The ONNX files must be exported from the matching ordinal checkpoints.
MODEL_FILES = {
    'resnet50': [
        'resnet50_ordinal_finetune_layer3_4_best_21092026.pth',
        'resnet50_ordinal_kl.onnx',
    ],
    'densenet121': [
        'densenet121_ordinal_finetune_block3_4_best_21092026.pth',
        'densenet121_ordinal_kl.onnx',
    ],
}
assert MODEL_NAME in MODEL_FILES
assert PROJECT_SOURCE.is_dir(), f'Project dataset not found: {PROJECT_SOURCE}'
assert MODELS_SOURCE.is_dir(), f'Model dataset not found: {MODELS_SOURCE}'
assert MANIFEST_SOURCE.is_file(), f'Manifest not found: {MANIFEST_SOURCE}'
assert EXPERIMENT_PROFILE in {'full', 'faithfulness_sensitivity', 'max_sensitivity_sensitivity'}
assert not (LIMIT is not None and SAMPLES_PER_KL is not None)
print('Selected model:', MODEL_NAME)
print('Experiment profile:', EXPERIMENT_PROFILE)
print('Manifest:', MANIFEST_SOURCE)

### Manifest note

The runner validates every image before calculation. If your real manifest is in a separate Kaggle dataset, either use **absolute Kaggle paths** in `image_path`, or copy its images under the project directory and use paths relative to it. Do not create labels from directory names.

In [ ]:
import shutil

PROJECT_DIR = Path('/kaggle/working/knee-osteoarthritis-xai')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(PROJECT_SOURCE, PROJECT_DIR, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__'))

models_dir = PROJECT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
for filename in MODEL_FILES[MODEL_NAME]:
    source = MODELS_SOURCE / filename
    if not source.is_file():
        raise FileNotFoundError(f'Missing required {MODEL_NAME} model file: {source}')
    shutil.copy2(source, models_dir / filename)

# A source manifest inside the project keeps its relative paths valid after copying.
# External absolute paths remain valid because Kaggle input datasets are mounted read-only.
if MANIFEST_SOURCE.is_relative_to(PROJECT_SOURCE):
    manifest_path = PROJECT_DIR / MANIFEST_SOURCE.relative_to(PROJECT_SOURCE)
else:
    manifest_path = MANIFEST_SOURCE

print('Project staged at:', PROJECT_DIR)
print('Results will be written under: /kaggle/working')

In [ ]:
# Run each configuration into its own CSV. Do not merge parameter configurations.
import subprocess

if EXPERIMENT_PROFILE == 'faithfulness_sensitivity':
    experiment_runs = [
        {'suffix': 'faithfulness-224', 'metrics': ['faithfulness-correlation'], 'subset_size': 224},
        {'suffix': 'faithfulness-1024', 'metrics': ['faithfulness-correlation'], 'subset_size': 1024},
        {'suffix': 'faithfulness-2048', 'metrics': ['faithfulness-correlation'], 'subset_size': 2048},
    ]
elif EXPERIMENT_PROFILE == 'max_sensitivity_sensitivity':
    experiment_runs = [
        {'suffix': 'maxsensitivity-raw', 'metrics': ['max-sensitivity'], 'normalise': False},
        {'suffix': 'maxsensitivity-normalised', 'metrics': ['max-sensitivity'], 'normalise': True},
    ]
else:
    experiment_runs = [
        {
            'suffix': 'full',
            'metrics': ['faithfulness-correlation', 'region-perturbation', 'max-sensitivity', 'sparseness', 'stability'],
            'subset_size': FINAL_FAITHFULNESS_SUBSET_SIZE,
            'normalise': False,
        }
    ]

output_paths = []
for config in experiment_runs:
    run_id = f'{RUN_ID_PREFIX}-{config["suffix"]}'
    output_path = Path('/kaggle/working') / f'{run_id}.csv'
    command = [
        sys.executable, 'scripts/run_research.py',
        '--manifest', str(manifest_path),
        '--model', MODEL_NAME,
        '--methods', *METHODS,
        '--output', str(output_path),
        '--run-id', run_id,
        '--seed', str(SEED),
        '--faithfulness-runs', str(FAITHFULNESS_RUNS),
        '--max-sensitivity-samples', str(MAX_SENSITIVITY_SAMPLES),
        '--metrics', *config['metrics'],
        '--device', 'cuda',
    ]
    if 'subset_size' in config:
        command += ['--faithfulness-subset-size', str(config['subset_size'])]
    if config.get('normalise', False):
        command.append('--max-sensitivity-normalise')
    else:
        command.append('--no-max-sensitivity-normalise')
    if LIMIT is not None:
        command += ['--limit', str(LIMIT)]
    if SAMPLES_PER_KL is not None:
        command += ['--samples-per-kl', str(SAMPLES_PER_KL)]
    if RESUME:
        command.append('--resume')

    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    output_paths.append(output_path)

print('Completed:', *output_paths, sep='\n- ' )

In [ ]:
# Inspect and download the validated results.
import pandas as pd

for output_path in output_paths:
    results = pd.read_csv(output_path)
    print(f'\n{output_path.name}: {len(results)} rows')
    display(results.groupby(['metric_category', 'metric_name']).size().rename('rows').reset_index())
    display(results.groupby(['xai_method'])['metric_value'].agg(['count', 'median', 'mean']).reset_index())
    if 'attribution_l2_norm' in results:
        display(results.groupby('xai_method')['attribution_l2_norm'].agg(['median', 'min', 'max']).reset_index())

print('In the right Kaggle panel, use Output → Save Version or download this CSV and place it in results/research/ locally.')